# 06 — Ensemble & full comparison

Every strategy, on one universe, one period, one engine — then blended.

Comparing them is only meaningful because they share all three. The rankings
below still say more about this particular sample than about the strategies.

Standalone: no `portfolio_agent` import. Needs `torch` for the two learned members.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# Simulation settings, shared by every notebook so the strategies are comparable.
#
# execution_lag=1 is the property that keeps this honest: a signal computed from
# day t's close is traded into day t+1's return. The engine refuses lag=0.
config = L.BacktestConfig(
    initial_capital=1_000_000.0,
    cost_bps=25.0,        # all-in round trip for Indian cash equities
    max_weight=0.10,
    rebalance_days=5,     # weekly; the main control over turnover
    max_gross=1.0,        # long-only, unlevered
    execution_lag=1,
)

benchmark = L.equal_weight_benchmark(close, config)
print("equal-weight buy & hold:",
      {k: round(v, 4) for k, v in benchmark.stats.items()
       if k in ("cagr", "sharpe", "max_drawdown")})

## Build every member

The two learned members are trained here on the same panel. Keep the epoch counts
modest — this cell does two full training runs.

In [ ]:
members = {}

members["rule_based"] = L.rule_based_scores(feature_panel, panel, use_monte_carlo=False)
members["momentum"] = L.momentum_scores(feature_panel, top_fraction=0.25)
members["low_volatility"] = L.low_volatility_scores(feature_panel, top_fraction=0.25)

if L.TORCH_AVAILABLE:
    supervised = L.build_supervised_panel(feature_panel, close,
                                          sequence_length=30, horizon=5)
    trained = L.train_lstm(supervised, epochs=25, patience=5, verbose=False)
    members["lstm"] = L.lstm_scores(trained, feature_panel, close, top_fraction=0.25)
    print(f"lstm: best epoch {trained['best_epoch']}, "
          f"val loss {trained['best_val_loss']:.5f}")

    sac = L.train_sac(feature_panel, close, epochs=25, gradient_steps=100, verbose=False)
    members["sac"] = L.sac_scores(sac, feature_panel, close, threshold=0.60)
    print("sac: trained")

print("\nmembers:", list(members))

## Blend

Members are put on a common scale per date before weighting. Without that, a
member whose scores happen to be large (inverse volatility runs to tens) drowns
one whose scores are bounded in [0, 1], and the configured weights describe
something other than what the blend actually does.

In [ ]:
weights = {name: 1.0 / len(members) for name in members}   # start equal
ensemble_scores = L.ensemble_scores(members, weights)

results = {name: L.run_backtest(scores, close, config)
           for name, scores in members.items()}
results["ensemble"] = L.run_backtest(ensemble_scores, close, config)
results["equal weight"] = benchmark

table = L.compare_stats(results)
display(table[["sharpe", "sortino", "cagr", "max_drawdown", "calmar",
               "avg_positions", "ann_turnover"]])

## Comparison

In [ ]:
L.plot_equity(results, title="All strategies", benchmark=benchmark.returns)

In [ ]:
L.plot_stats_table(table, title="Risk-adjusted comparison")

In [ ]:
# Correlation of daily returns. An ensemble only diversifies to the extent its
# members disagree — highly correlated members give the blend of one strategy
# with more turnover.
import matplotlib.pyplot as plt

daily = pd.DataFrame({name: result.returns for name, result in results.items()})
correlation = daily.drop(columns=["equal weight"], errors="ignore").corr()

fig, ax = plt.subplots(figsize=(6.5, 5.2))
image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation))); ax.set_xticklabels(correlation.columns,
                                                           rotation=45, ha="right")
ax.set_yticks(range(len(correlation))); ax.set_yticklabels(correlation.index)
for i in range(len(correlation)):
    for j in range(len(correlation)):
        ax.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="black")
ax.set_title("Strategy return correlation"); ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# Rolling 6-month Sharpe: which member was carrying the blend, and when. A
# strategy that is only ever good in one regime shows up here and nowhere else.
window = 126
rolling = daily.rolling(window).mean() / daily.rolling(window).std() * np.sqrt(252)

ax = rolling.plot(figsize=(12, 4.2), linewidth=1.1)
ax.axhline(0, color="#444444", linewidth=1)
ax.set_title(f"Rolling {window}-day Sharpe by strategy")
ax.grid(**L.GRID); ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

## Weighting the blend

Equal weights are a defensible default and a poor optimum. Anything fitted on
this sample is fitted on the sample — the sweep below is here to show the
*spread* of outcomes across weightings, not to pick the best one.

In [ ]:
schemes = {"equal": {name: 1 / len(members) for name in members}}
if "momentum" in members and "low_volatility" in members:
    schemes["momentum tilt"] = {
        name: (0.5 if name == "momentum" else 0.5 / (len(members) - 1))
        for name in members
    }
    schemes["defensive tilt"] = {
        name: (0.5 if name == "low_volatility" else 0.5 / (len(members) - 1))
        for name in members
    }

sweep = {
    label: L.run_backtest(L.ensemble_scores(members, scheme), close, config)
    for label, scheme in schemes.items()
}
sweep["best single member"] = max(
    (results[name] for name in members),
    key=lambda r: r.stats.get("sharpe", -99),
)
display(L.compare_stats(sweep)[["sharpe", "cagr", "max_drawdown", "ann_turnover"]])

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.